# Taxonomy Extraction

Run the enriched taxonomy prompt on papers predicted as POSITIVE by the classifier.
Uses the `taxonomy` task config from `llm_labeller.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve().parent))

In [2]:
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

In [ ]:
from src.labelling.llm_labeller import (
    claude_batch_submit,
    claude_batch_results,
    label_papers,
    label_papers_async,
)

## 1. Load positive predictions

In [4]:
predictions = pd.read_parquet(
    DATA_DIR / "predictions" / "inference_predictions_base.parquet"
)

# Keep only predicted positives
positives = predictions[predictions["predicted"] == 1].copy()
print(f"Positive predictions: {len(positives):,} papers")


Positive predictions: 3,738 papers


In [17]:
anthology = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")

# Filter to positives in the anthology
positives_paper = anthology[anthology["bibkey"].isin(positives["bibkey"])]
# paper avant 2020
positives_paper = positives_paper[positives_paper["year"] < 2021]
print(f"Positive predictions in anthology: {len(positives_paper):,} papers")

Positive predictions in anthology: 1,170 papers


## 2. Submit Claude batch (taxonomy task)

In [ ]:
job_id = claude_batch_submit(positives_paper, task="taxonomy")
job_id

In [ ]:
job_id = "msgbatch_01BE6FuJZqzPcAZfBzy7hYFV"

In [ ]:
# DeepSeek — async concurrent (no batch API available)
results_deepseek = await label_papers_async(
    positives_paper,
    provider="deepseek",
    task="taxonomy",
    max_concurrent=20,
    checkpoint_path=str(DATA_DIR / "labeled" / "deepseek_taxonomy_checkpoint.parquet"),
)
results_deepseek.head()

In [10]:
results_deepseek.to_parquet(
    DATA_DIR / "labeled" / "deepseek_taxonomy.parquet", index=False
)

## 3. Retrieve results

In [ ]:
results = claude_batch_results(job_id, positives_paper, task="taxonomy")


In [ ]:
results.to_parquet(DATA_DIR / "labeled" / "taxonomy_claude_1.parquet", index=False)
results.head()

## Analyse

In [11]:
results_deepseek = pd.read_parquet(DATA_DIR / "labeled" / "deepseek_taxonomy.parquet")
results_claude = pd.read_parquet(DATA_DIR / "labeled" / "taxonomy_claude_1.parquet")

In [14]:
results_merged_claude_deepseek = results_claude.merge(
    results_deepseek[["bibkey", "llm_label", "llm_justification"]], on="bibkey"
)